# Fair Value Research Model

This notebook is the fair-value layer of the market-making stack. It estimates a short-horizon fair value from point-in-time market state features, then translates that estimate into an edge in basis points and a fair-value price band.

This is not a quoting strategy by itself. A market maker still needs fill-probability, adverse-selection, inventory, fee, latency, and risk controls before placing quotes.

## Research Definition

Let `S_t` be the current reference price and `x_t` be the feature vector known at time `t`. The model estimates the expected executable log return over a short market-making horizon:

$$
y_{t,h} = 10^4 \cdot \log\left(\frac{P^{\mathrm{exec}}_{t+h}}{S_t}\right)
$$

$$
\widehat{y}_{t,h} = f_\theta(x_t)
$$

The fair value implied by the model is:

$$
\widehat{F}_{t,h} = S_t \exp\left(\frac{\widehat{y}_{t,h}}{10^4}\right)
$$

The model edge is simply:

$$
\widehat{e}_{t,h}^{\mathrm{bps}} = 10^4 \cdot \log\left(\frac{\widehat{F}_{t,h}}{S_t}\right) = \widehat{y}_{t,h}
$$

For a downstream inventory-aware quoting model, fair value becomes the center input. A common reservation-price form is:

$$
r_t = \widehat{F}_{t,h} - q_t \gamma \widehat{\sigma}_t^2 \tau
$$

where `q_t` is inventory, `gamma` is risk aversion, `sigma_t` is volatility, and `tau` is the control horizon. This notebook estimates `F`; it does not optimize `q`, spread, or quote placement.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if "notebooks" not in sys.path:
    sys.path.append("notebooks")

from advanced_features import (
    FEATURE_DATASET_KEYS,
    available_dataset_dates,
    build_feature_set,
    build_targets,
    discover_datasets,
)

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
HORIZONS = tuple(int(value) for value in os.environ.get("MODL_HORIZONS", "5,15,30").split(","))
DATE_FILTER = [value.strip() for value in os.environ.get("MODL_FAIR_VALUE_DATES", "").split(",") if value.strip()]

INCLUDE_PARTIAL_DATES = True
FAIR_VALUE_TARGET = os.environ.get("MODL_FAIR_VALUE_TARGET", "target_fir_return_entry5m_wait0m_exit30m")
MIN_FEATURE_OBS = int(os.environ.get("MODL_FAIR_VALUE_MIN_FEATURE_OBS", "50"))
MIN_TRAIN_OBS = int(os.environ.get("MODL_FAIR_VALUE_MIN_TRAIN_OBS", "150"))
RIDGE_ALPHA = float(os.environ.get("MODL_FAIR_VALUE_RIDGE_ALPHA", "25.0"))
BAND_Z = float(os.environ.get("MODL_FAIR_VALUE_BAND_Z", "1.0"))
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 240)
pd.set_option("display.max_colwidth", 220)

ROOT, BAR_MINUTES, HORIZONS, FAIR_VALUE_TARGET

## Date Inventory And Panel Build

Each date is built independently. This prevents rolling features and future targets from leaking across date boundaries. Partial dates are included and explicitly flagged.

In [ ]:
date_key_map = available_dataset_dates(ROOT)
date_inventory = pd.DataFrame(
    [
        {
            "date": date,
            "dataset_count": len(keys),
            "missing_required": sorted(set(FEATURE_DATASET_KEYS) - set(keys)),
            "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(keys),
        }
        for date, keys in sorted(date_key_map.items())
    ]
)
if DATE_FILTER:
    date_inventory = date_inventory[date_inventory["date"].isin(DATE_FILTER)].copy()
if not INCLUDE_PARTIAL_DATES:
    date_inventory = date_inventory[date_inventory["is_full_dataset"]].copy()

feature_by_date: dict[str, pd.DataFrame] = {}
target_by_date: dict[str, pd.DataFrame] = {}
model_by_date: dict[str, pd.DataFrame] = {}
build_rows: list[dict[str, object]] = []
build_errors: list[dict[str, object]] = []

for date in date_inventory["date"].tolist():
    date_tag = datetime.strptime(date, "%Y-%m-%d").strftime("%y-%m-%d")
    datasets = discover_datasets(ROOT, date_tag)
    try:
        feature_set = build_feature_set(datasets, horizons=HORIZONS, bar_minutes=BAR_MINUTES)
        features = feature_set.feature_matrix.copy()
        targets = build_targets(feature_set.reference_price, feature_set.term_structure, HORIZONS, bar_minutes=BAR_MINUTES)
        model = pd.concat([features, targets], axis=1).sort_index()

        feature_by_date[date] = features
        target_by_date[date] = targets
        model_by_date[date] = model
        build_rows.append(
            {
                "date": date,
                "is_full_dataset": set(FEATURE_DATASET_KEYS).issubset(date_key_map[date]),
                "datasets": len(datasets),
                "feature_rows": len(features),
                "feature_columns": features.shape[1],
                "target_rows": len(targets),
                "target_columns": targets.shape[1],
                "reference_obs": int(feature_set.reference_price.notna().sum()),
            }
        )
    except Exception as exc:  # noqa: BLE001 - research notebook should report and continue
        build_errors.append({"date": date, "error_type": type(exc).__name__, "error": str(exc)})

if not model_by_date:
    raise RuntimeError("No fair-value dates built successfully")

build_summary = pd.DataFrame(build_rows).sort_values("date")
build_errors = pd.DataFrame(build_errors)
feature_panel = pd.concat(feature_by_date, names=["date", "minute"]).sort_index()
target_panel = pd.concat(target_by_date, names=["date", "minute"]).sort_index()
model_panel = pd.concat(model_by_date, names=["date", "minute"]).sort_index()

display(date_inventory)
display(build_summary)
if not build_errors.empty:
    display(build_errors)
model_panel.shape

## Feature Policy

The fair-value model uses market state and microstructure variables. It excludes direct price levels such as raw mids and VWAPs because those can dominate the regression without adding fair-value edge. Relative price dislocations, basis, funding, flow, volatility, and order-book state are allowed.

In [ ]:
ROLLING_SUFFIXES = (
    f"_diff_{BAR_MINUTES}m",
    "_mean_5m",
    "_z_5m",
    "_mean_15m",
    "_z_15m",
    "_mean_30m",
    "_z_30m",
)


def base_feature_name(feature: str) -> str:
    for suffix in ROLLING_SUFFIXES:
        if feature.endswith(suffix):
            return feature[: -len(suffix)]
    return feature


def feature_family(feature: str) -> str:
    base = base_feature_name(feature)
    rules = [
        ("hawkes_bsi", ("hawkes_bsi_",)),
        ("trade_flow", ("trade_flow_imbalance_",)),
        ("trade_activity", ("trade_volume_", "trade_trade_count_", "trade_vwap_")),
        ("book_microstructure", ("book_",)),
        ("cross_venue", ("cross_",)),
        ("spread_mid_arbitrage", ("mid_", "spread_")),
        ("realized_vol", ("rv_", "bpv_", "jump_")),
        ("returns", ("ret_",)),
        ("term_structure", ("iv_", "term_", "short_iv_decimal")),
        ("option_smile", ("smile_",)),
        ("futures_basis", ("basis_",)),
        ("funding", ("estimated_funding_rate",)),
    ]
    for family, prefixes in rules:
        if base.startswith(prefixes):
            return family
    return "other"


def is_disallowed_price_level(feature: str) -> bool:
    base = base_feature_name(feature)
    if base in {"reference_price", "log_price", "median_index_price", "mid_hibachi_minus_hyperliquid"}:
        return True
    if base.startswith(("book_mid_", "trade_vwap_")):
        return True
    if base.endswith(("_count", "_updates")) and not base.startswith("trade_trade_count_"):
        return True
    if base in {"active_options", "option_tick_count"}:
        return True
    return False


included_families = {
    "hawkes_bsi",
    "trade_flow",
    "trade_activity",
    "book_microstructure",
    "cross_venue",
    "spread_mid_arbitrage",
    "realized_vol",
    "returns",
    "term_structure",
    "option_smile",
    "futures_basis",
    "funding",
}

numeric_features = [column for column in feature_panel.select_dtypes(include=[np.number]).columns if not column.startswith("target_")]
candidate_features = [
    column
    for column in numeric_features
    if feature_family(column) in included_families
    and not is_disallowed_price_level(column)
    and feature_panel[column].replace([np.inf, -np.inf], np.nan).notna().sum() >= MIN_FEATURE_OBS
    and feature_panel[column].nunique(dropna=True) > 1
]

feature_catalog = pd.DataFrame(
    {
        "feature": candidate_features,
        "base_feature": [base_feature_name(column) for column in candidate_features],
        "family": [feature_family(column) for column in candidate_features],
        "is_rolling_transform": [base_feature_name(column) != column for column in candidate_features],
    }
)

display(feature_catalog.groupby("family").size().rename("feature_count").sort_values(ascending=False).to_frame())
display(feature_catalog.head(30))
len(candidate_features)

## Fair-Value Model

The default model is ridge regression on standardized features:

$$
\widehat{\beta} = \arg\min_\beta \sum_t (y_t - \beta_0 - z_t^\top \beta)^2 + \alpha \|\beta\|_2^2
$$

where `z_t` is the imputed and standardized feature vector. Ridge is intentionally simple and stable for small samples; it is a fair-value baseline before nonlinear models are justified.

In [ ]:
def make_fair_value_model(alpha: float = RIDGE_ALPHA) -> Pipeline:
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def valid_training_frame(frame: pd.DataFrame, target: str, features: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    available_features = [column for column in features if column in frame.columns]
    table = frame.reindex(columns=available_features + [target]).replace([np.inf, -np.inf], np.nan)
    table = table.dropna(subset=[target])
    usable_features = [column for column in features if table[column].notna().sum() >= MIN_FEATURE_OBS and table[column].nunique(dropna=True) > 1]
    return table[usable_features], table[target].mul(10_000.0)


def regression_metrics(y_true_bps: pd.Series, y_pred_bps: pd.Series) -> dict[str, float]:
    pair = pd.concat([y_true_bps.rename("actual"), y_pred_bps.rename("pred")], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if pair.empty:
        return {"n": 0, "rmse_bps": np.nan, "mae_bps": np.nan, "spearman_ic": np.nan, "pearson_corr": np.nan, "directional_hit_rate": np.nan}
    err = pair["pred"] - pair["actual"]
    nonzero = pair[(pair["actual"] != 0) & (pair["pred"] != 0)]
    hit_rate = np.nan if nonzero.empty else (np.sign(nonzero["actual"]) == np.sign(nonzero["pred"])).mean()
    return {
        "n": int(len(pair)),
        "rmse_bps": float(np.sqrt(np.mean(np.square(err)))),
        "mae_bps": float(np.mean(np.abs(err))),
        "spearman_ic": float(pair["actual"].corr(pair["pred"], method="spearman")) if pair["actual"].nunique() > 1 and pair["pred"].nunique() > 1 else np.nan,
        "pearson_corr": float(pair["actual"].corr(pair["pred"], method="pearson")) if pair["actual"].nunique() > 1 and pair["pred"].nunique() > 1 else np.nan,
        "directional_hit_rate": float(hit_rate) if pd.notna(hit_rate) else np.nan,
    }


if FAIR_VALUE_TARGET not in model_panel.columns:
    raise KeyError(f"FAIR_VALUE_TARGET is missing: {FAIR_VALUE_TARGET}")


## Walk-Forward Fair Value

For each date, the walk-forward model trains only on prior dates. Dates without enough prior training observations are skipped. With the current data, this is deliberately strict: early dates are included in the panel but may not receive an honest out-of-sample prediction.

In [ ]:
walk_forward_predictions: list[pd.DataFrame] = []
walk_forward_rows: list[dict[str, object]] = []
ordered_dates = sorted(model_by_date)

for test_date in ordered_dates:
    train_dates = [date for date in ordered_dates if date < test_date]
    if not train_dates:
        walk_forward_rows.append({"test_date": test_date, "status": "skipped_no_prior_dates", "train_obs": 0, "test_obs": len(model_by_date[test_date])})
        continue

    train_frame = pd.concat([model_by_date[date] for date in train_dates]).sort_index()
    test_frame = model_by_date[test_date].copy()
    train_x, train_y = valid_training_frame(train_frame, FAIR_VALUE_TARGET, candidate_features)
    if len(train_y) < MIN_TRAIN_OBS or train_x.shape[1] == 0:
        walk_forward_rows.append({"test_date": test_date, "status": "skipped_insufficient_train", "train_obs": len(train_y), "test_obs": len(test_frame)})
        continue

    model = make_fair_value_model()
    model.fit(train_x, train_y)
    train_pred = pd.Series(model.predict(train_x), index=train_x.index, name="train_pred_bps")
    train_resid_sigma = float((train_y - train_pred).std())

    test_x = test_frame.reindex(columns=train_x.columns).replace([np.inf, -np.inf], np.nan)
    pred = pd.Series(model.predict(test_x), index=test_x.index, name="walk_forward_edge_bps")
    actual = test_frame[FAIR_VALUE_TARGET].mul(10_000.0).rename("actual_target_bps")
    metrics = regression_metrics(actual, pred)

    out = pd.DataFrame(index=pd.MultiIndex.from_product([[test_date], test_frame.index], names=["date", "minute"]))
    out["walk_forward_edge_bps"] = pred.to_numpy()
    out["walk_forward_sigma_bps"] = train_resid_sigma
    walk_forward_predictions.append(out)

    walk_forward_rows.append(
        {
            "test_date": test_date,
            "status": "fit",
            "train_dates": ",".join(train_dates),
            "train_obs": len(train_y),
            "test_obs": len(test_frame),
            "feature_count": train_x.shape[1],
            "train_resid_sigma_bps": train_resid_sigma,
            **metrics,
        }
    )

walk_forward_summary = pd.DataFrame(walk_forward_rows)
walk_forward_panel = pd.concat(walk_forward_predictions).sort_index() if walk_forward_predictions else pd.DataFrame(index=model_panel.index)
display(walk_forward_summary)
walk_forward_panel.tail(20)

## All-Sample Research Fit

This fit uses every available date and is therefore not an out-of-sample performance estimate. It is useful for inspecting coefficients, signs, feature families, and fair-value shape. Production training must be walk-forward or use an explicitly frozen training window.

In [ ]:
research_x, research_y = valid_training_frame(model_panel, FAIR_VALUE_TARGET, candidate_features)
research_model = make_fair_value_model()
research_model.fit(research_x, research_y)
research_pred = pd.Series(research_model.predict(model_panel[research_x.columns].replace([np.inf, -np.inf], np.nan)), index=model_panel.index, name="research_edge_bps")
research_actual = model_panel[FAIR_VALUE_TARGET].mul(10_000.0).rename("actual_target_bps")
research_metrics = pd.DataFrame([regression_metrics(research_actual, research_pred)])
research_sigma_bps = float((research_actual.reindex(research_pred.index) - research_pred).dropna().std())

ridge = research_model.named_steps["ridge"]
coef_table = pd.DataFrame(
    {
        "feature": research_x.columns,
        "coef_standardized_bps": ridge.coef_,
    }
)
coef_table["abs_coef"] = coef_table["coef_standardized_bps"].abs()
coef_table["base_feature"] = coef_table["feature"].map(base_feature_name)
coef_table["family"] = coef_table["feature"].map(feature_family)
coef_table = coef_table.sort_values("abs_coef", ascending=False).reset_index(drop=True)
family_coef = coef_table.groupby("family")["abs_coef"].sum().sort_values(ascending=False).rename("sum_abs_standardized_coef").to_frame()

display(research_metrics)
display(family_coef)
display(coef_table.head(40))

## Fair-Value Table

The fair-value table converts edge estimates into price levels and uncertainty bands. The band is a residual-volatility diagnostic, not a calibrated execution probability.

In [ ]:
fair_value_table = pd.DataFrame(index=model_panel.index)
fair_value_table["reference_price"] = model_panel["reference_price"]
fair_value_table["target_bps"] = model_panel[FAIR_VALUE_TARGET].mul(10_000.0)
fair_value_table["research_edge_bps"] = research_pred
fair_value_table["research_sigma_bps"] = research_sigma_bps

if not walk_forward_panel.empty:
    fair_value_table = fair_value_table.join(walk_forward_panel, how="left")

for prefix in ["research", "walk_forward"]:
    edge_col = f"{prefix}_edge_bps"
    sigma_col = f"{prefix}_sigma_bps"
    if edge_col not in fair_value_table.columns:
        continue
    fair_value_table[f"{prefix}_fair_value"] = fair_value_table["reference_price"] * np.exp(fair_value_table[edge_col] / 10_000.0)
    if sigma_col in fair_value_table.columns:
        fair_value_table[f"{prefix}_fair_value_low"] = fair_value_table["reference_price"] * np.exp((fair_value_table[edge_col] - BAND_Z * fair_value_table[sigma_col]) / 10_000.0)
        fair_value_table[f"{prefix}_fair_value_high"] = fair_value_table["reference_price"] * np.exp((fair_value_table[edge_col] + BAND_Z * fair_value_table[sigma_col]) / 10_000.0)

display(fair_value_table.tail(30))
fair_value_table.describe().T

## Diagnostics

The plots show edge in basis points and fair value around the reference price. The edge plot is usually more informative than price-level plots because BTC price levels make small fair-value shifts visually hard to see.

In [ ]:
latest_date = sorted(model_by_date)[-1]
latest = fair_value_table.xs(latest_date, level="date").copy()

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)
latest[["reference_price", "research_fair_value"]].plot(ax=axes[0])
if "walk_forward_fair_value" in latest.columns and latest["walk_forward_fair_value"].notna().any():
    latest["walk_forward_fair_value"].plot(ax=axes[0], label="walk_forward_fair_value")
axes[0].set_title(f"Reference Price And Fair Value: {latest_date}")
axes[0].legend()

edge_cols = [column for column in ["research_edge_bps", "walk_forward_edge_bps", "target_bps"] if column in latest.columns]
latest[edge_cols].plot(ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Fair-Value Edge And Target, bps")

if {"research_fair_value_low", "research_fair_value_high"}.issubset(latest.columns):
    axes[2].plot(latest.index, latest["reference_price"], label="reference_price")
    axes[2].plot(latest.index, latest["research_fair_value"], label="research_fair_value")
    axes[2].fill_between(latest.index, latest["research_fair_value_low"], latest["research_fair_value_high"], alpha=0.2, label="research +/- residual sigma")
    axes[2].set_title("Research Fair-Value Band")
    axes[2].legend()

fig.autofmt_xdate(); plt.tight_layout()

plt.figure(figsize=(12, max(5, 0.25 * min(30, len(coef_table)))))
sns.barplot(data=coef_table.head(30), y="feature", x="coef_standardized_bps", hue="family", dodge=False)
plt.axvline(0, color="black", linewidth=1)
plt.title("Top Standardized Fair-Value Coefficients")
plt.tight_layout()

## Portfolio Manager Interpretation

The fair-value edge is not a trade by itself. It answers: given current observable state, where should the short-horizon executable reference price be centered?

Use cases:

- Center passive quotes around fair value rather than raw mid.
- Widen quotes or reduce size when fair-value uncertainty is high.
- Skew bid/ask placement when fair value disagrees with the displayed mid.
- Feed the edge into inventory-aware reservation price logic.

Required next layers:

1. Fill model: estimate fill probability as a function of quote distance and queue state.
2. Adverse-selection model: estimate post-fill markout conditional on fill.
3. Inventory model: convert edge and risk into quote skew.
4. Event-driven backtest: include fees, latency, cancels, partial fills, queue priority, and venue constraints.

A quote should only be placed when expected spread capture plus fair-value edge exceeds expected adverse selection, fees, and inventory penalty.

## Optional Save

Set `SAVE_OUTPUTS = True` to write the fair-value table, walk-forward summary, and coefficient table.

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "fair_value" / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    date_inventory.to_parquet(out_dir / "date_inventory.parquet")
    build_summary.to_parquet(out_dir / "build_summary.parquet")
    walk_forward_summary.to_parquet(out_dir / "walk_forward_summary.parquet")
    fair_value_table.to_parquet(out_dir / "fair_value_table.parquet")
    coef_table.to_parquet(out_dir / "fair_value_coefficients.parquet")
    print(f"wrote fair-value outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")